In [0]:
from pyspark.sql.functions import col, coalesce, current_timestamp

In [0]:
Geolocation_df=spark.read.format("json").load("/Volumes/pricing_analytics/storageac/pricing_analytics/landing/Geo_location/")

In [0]:
last_processed_date=spark.sql("""select coalesce(max(process_table_date_time), date('1900-01-01')) as max_date from pricing_analytics.bronze.pipeline_control_logs where process_name='Geolocation_ingestion' and process_status='Completed'""")

In [0]:
final_df=Geolocation_df.crossJoin(last_processed_date).filter(col("File_ingestion_date")>col('max_date')).withColumn("Lake_house_ingestion_date", current_timestamp()).withColumn("lake_house_updated_date", current_timestamp()).drop('max_date')

In [0]:
final_df.write.format("delta").option("mergeSchema", True).mode("append").save("/Volumes/pricing_analytics/storageac/pricing_analytics/bronze/Geo_Location/")

In [0]:
final_df.write.format("delta").option("mergeSchema", True).mode("append").saveAsTable("pricing_analytics.bronze.Geo_location_data")

In [0]:
%sql
insert into pricing_analytics.bronze.pipeline_control_logs(
  select 
  'Geolocation_ingestion',
  max(File_ingestion_date),
  'Completed',
  current_timestamp()
  from pricing_analytics.bronze.geo_location_data
)